# Phase 6b: dialect tag and Standard Greek rehearsal for Rumble-2-Flash

Section 15 of `docs/PROJECT_LOG.md`: the Cypriot LoRA breaks Standard Greek (FLEURS 14.1% → 39.4%). Section 16:
weight blending trades one against the other but cannot fix both. Three LoRA variants (`VARIANTS` below), which
with the plain LoRA of section 11 make a 2 × 2 design (tag yes/no × Standard Greek data yes/no):
- `tag`: the tag only, no Standard Greek data (`finetune_rumble-2-flash_tag.yaml`). No extra download needed.
- `mix_tag`: tag + rehearsal, as described next (`finetune_rumble-2-flash_mix_tag.yaml`).
- `mix`: rehearsal without the tag (`finetune_rumble-2-flash_mix.yaml`).

The ingredients:
- **Rehearsal**: 410 Common Voice Greek clips (Standard Greek, `scripts/prep_cv_greek.py`) mixed into the 820
  Cypriot pieces, so about 1/3 of each batch is Standard Greek.
- **Dialect tag**: Cypriot pieces get "Cypriot Greek" as system context after Rumble's control tokens; Standard
  Greek pieces get none. Untagged inference should therefore keep standard output.
- Same recipe otherwise (`configs/finetune_rumble-2-flash_mix_tag.yaml`); cheaper dev checks (every 250 steps,
  150 Cypriot pieces + 100 Common Voice dev clips).

**Evaluation** (all on this GPU, bf16, batch 1):
1. Cypriot dev, 280 full clips, **with** the tag (the main Cypriot number; compare 17.3% for the plain LoRA).
2. FLEURS `el_gr`, same 200 clips as notebooks 09/10, **without** the tag (the main Standard Greek number), plus
   zero-shot Rumble and the plain LoRA re-run here so all FLEURS rows share GPU and dtype.
3. Diagnostics: Cypriot dev without the tag and FLEURS with the tag (does the tag switch the spelling?).
Steps 1-2 run for every variant; step 3 for the tagged ones.

Runtime: **L4** (the memory settings were tuned on it). About 2 h per tagged variant (training ~50 min, Cypriot dev
twice ~28 min each, FLEURS a few minutes per run), ~1.5 h for `mix`. Everything runs in the background, writes to
Drive and releases the GPU at the end. Needs on Drive in `MyDrive/cyasr/`: `cyasr_bundle.zip`, `cyasr_code.zip`
(current), `segments.zip`, and `segments_cv_el.zip` for the `mix` variants.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
CY = '/content/drive/MyDrive/cyasr'
DRIVE_RUNS = f'{CY}/runs'
OUT = f'{CY}/runs_mix'            # FLEURS results of this notebook
VARIANTS = ['tag']                # any of 'tag', 'mix_tag', 'mix' (the mix ones need segments_cv_el.zip)

!rm -rf /content/cyasr && mkdir -p /content/cyasr
!unzip -q {CY}/cyasr_bundle.zip -d /content/cyasr
!unzip -q -o {CY}/cyasr_code.zip -d /content/cyasr
%cd /content/cyasr
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao
!cp {CY}/segments.zip /content/ && mkdir -p data && unzip -q -o /content/segments.zip -d data/
!ls data/segments/train | wc -l       # expect 820
if any(v.startswith('mix') for v in VARIANTS):
    !cp {CY}/segments_cv_el.zip /content/ && unzip -q -o /content/segments_cv_el.zip -d data/
    !ls data/segments_cv_el/train | wc -l # expect 410
!ls scripts/eval_fleurs.py configs/finetune_rumble-2-flash_tag.yaml   # fails if cyasr_code.zip is old
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Cache the FLEURS el_gr clips (same loader and the same first 200 test clips as notebooks 09 and 10)
import csv, io, os, pickle, tarfile, requests, soundfile as sf
FLEURS = 'https://huggingface.co/datasets/google/fleurs/resolve/main/data'
def load_fleurs(lang, n=None):
    tsv = requests.get(f'{FLEURS}/{lang}/test.tsv', timeout=60); tsv.raise_for_status()
    refs = {}
    for row in csv.reader(io.StringIO(tsv.text), delimiter='\t', quoting=csv.QUOTE_NONE):
        refs.setdefault(row[1], (row[0], row[2]))
    clips = []
    with requests.get(f'{FLEURS}/{lang}/audio/test.tar.gz', stream=True, timeout=60) as r:
        r.raise_for_status(); r.raw.decode_content = True
        with tarfile.open(fileobj=r.raw, mode='r|gz') as tar:
            for m in tar:
                name = os.path.basename(m.name)
                if not m.isfile() or name not in refs:
                    continue
                audio, sr = sf.read(io.BytesIO(tar.extractfile(m).read()), dtype='float32'); assert sr == 16000
                uid, ref = refs[name]
                clips.append({'audio_id': name, 'utt_id': uid, 'ref': ref, 'audio': audio, 'duration_s': len(audio) / sr})
                if n and len(clips) >= n:
                    break
    return clips
FL = '/content/fleurs_el_gr_200.pkl'
if not os.path.exists(FL):
    pickle.dump(load_fleurs('el_gr', 200), open(FL, 'wb'))
print(len(pickle.load(open(FL, 'rb'))), 'FLEURS clips cached')

In [ ]:
# Smoke test of the first variant: 4 steps on a few pieces. Check the example input(s): a Cypriot one has
# 'Cypriot Greek' after <nodigits><filler><verbatim> (tagged variants); a Standard Greek one (mix variants) has
# nothing there. Also check peak_gpu_memory_gb.
import yaml, json, time, subprocess
!rm -rf runs/finetune_smoke
cfg = yaml.safe_load(open(f'configs/finetune_rumble-2-flash_{VARIANTS[0]}.yaml'))
cfg.update(run_name='smoke', max_steps=4, eval_steps=2, warmup_steps=1, eval_subset=8, smg_dev_samples=8)
yaml.safe_dump(cfg, open('/content/smoke.yaml', 'w'), allow_unicode=True)
!python -m cyasr.train --config /content/smoke.yaml > /content/smoke.log 2>&1
!grep -E -A6 "example input" /content/smoke.log | head -20
!grep -E "training mix|eval_cy_wer|eval_smg_wer|timing|Error" /content/smoke.log | tail -5

In [ ]:
# The real runs in the background: train, then evaluate. Resume is on and checkpoints are on Drive, so after a
# disconnect rerun the setup cells and this one. Log on Drive.
LOG = f'{DRIVE_RUNS}/rumble_variants.log'
F = 'grep --line-buffered -v -i -E "warn|Loading weights|it/s]"'
RUMBLE = 'Reson8/Rumble-2-Flash --revision 95bd5f9133a2398e5e48a98c089fe48e122bde97'
def cfgs(variant, tag):
    tr = yaml.safe_load(open(f'configs/finetune_rumble-2-flash_{variant}.yaml')); tr.update(output_dir=DRIVE_RUNS, resume=True)
    yaml.safe_dump(tr, open(f'/content/train_{variant}.yaml', 'w'), allow_unicode=True)
    ev = yaml.safe_load(open(f'configs/eval_rumble-2-flash_{variant}.yaml')); ev.update(output_dir=DRIVE_RUNS)
    yaml.safe_dump(ev, open(f'/content/eval_{variant}.yaml', 'w'), allow_unicode=True)
    if tag:   # diagnostic: the same model on Cypriot dev without the tag
        ev.update(prompt=None, run_name=ev['run_name'] + '_notag')
        yaml.safe_dump(ev, open(f'/content/eval_{variant}_notag.yaml', 'w'), allow_unicode=True)
    return f'{DRIVE_RUNS}/finetune_rumble-2-flash_{variant}/best'
steps = [('fleurs zero-shot', f'python scripts/eval_fleurs.py --model {RUMBLE} --name zeroshot --fleurs {FL} --out {OUT}'),
         ('fleurs plain lora', f'python scripts/eval_fleurs.py --model {DRIVE_RUNS}/finetune_rumble-2-flash_lora/best --name lora --fleurs {FL} --out {OUT}')]
for v in VARIANTS:
    tagged = v != 'mix'
    best = cfgs(v, tagged)
    steps += [(f'train {v}', f'python -m cyasr.train --config /content/train_{v}.yaml'),
              (f'cy dev {v}', f'python -m cyasr.run_baseline --config /content/eval_{v}.yaml'),
              (f'fleurs {v} (no tag)', f'python scripts/eval_fleurs.py --model {best} --name {v}_notag --fleurs {FL} --out {OUT}')]
    if tagged:   # diagnostics: does the tag switch the spelling?
        steps += [(f'fleurs {v} (tag)', f'python scripts/eval_fleurs.py --model {best} --name {v}_tag --prompt "Cypriot Greek" --fleurs {FL} --out {OUT}'),
                  (f'cy dev {v} (no tag)', f'python -m cyasr.run_baseline --config /content/eval_{v}_notag.yaml')]
sh = ''.join(f'echo ">>> {n} start $(date +%T)"\n{c} 2>&1 | {F}\necho ">>> {n} end $(date +%T)"\n' for n, c in steps)
open('/content/run_variants.sh', 'w').write(sh + 'echo ">>> ALL DONE"\n')
!mkdir -p {DRIVE_RUNS} {OUT}
proc = subprocess.Popen(f'cd /content/cyasr && PYTHONUNBUFFERED=1 nohup bash /content/run_variants.sh > {LOG} 2>&1', shell=True)
print('started; log:', LOG)

In [ ]:
# Progress (rerun any time)
!grep -E ">>>|eval_cy_wer_basic|eval_smg_wer_basic|\[dev\] [0-9]+/280|WER raw|FLEURS WER|Error|Traceback" {LOG} | tail -10; date +%T

In [ ]:
# Waits for everything, prints the results, then releases the GPU if AUTO_DISCONNECT.
AUTO_DISCONNECT = True
running = lambda: subprocess.run('pgrep -f "[r]un_variants.sh"', shell=True).returncode == 0
while running():
    time.sleep(120)
try:
    for v in VARIANTS:
        log = json.load(open(f'{DRIVE_RUNS}/finetune_rumble-2-flash_{v}/train_log.json'))
        loss = {e['step']: e['loss'] for e in log if 'loss' in e}
        print(f'\n{v}: step  train_loss  cy_WER  cy_CER_nodiac  smg(CV dev)_WER   (cy = 150 dev pieces)')
        k = 'eval_cy_' if any('eval_cy_wer_basic' in e for e in log) else 'eval_'   # mix variants log cy_ / smg_
        cy = {e['step']: {kk.replace(k, 'eval_cy_'): vv for kk, vv in e.items()} for e in log if f'{k}wer_basic' in e}
        smg = {e['step']: e for e in log if 'eval_smg_wer_basic' in e}
        for s in sorted(cy):
            print(f"  {s:5} {loss.get(s, float('nan')):10.3f} {cy[s]['eval_cy_wer_basic']:7.1%} {cy[s]['eval_cy_cer_nodiac']:9.1%} "
                  f"{smg[s]['eval_smg_wer_basic'] if s in smg else float('nan'):12.1%}")
        print(' ', json.dumps(json.load(open(f'{DRIVE_RUNS}/finetune_rumble-2-flash_{v}/timing.json'))))
    def cyd(run):
        p = f'{DRIVE_RUNS}/{run}/metrics.json'
        if not os.path.exists(p): return None
        lv = json.load(open(p))['levels']; return lv['basic']['wer'], lv['nodiac']['wer'], lv['nodiac']['cer']
    def fl(name):
        p = f'{OUT}/{name}/metrics.json'
        if not os.path.exists(p): return None
        m = json.load(open(p)); return m['levels']['basic']['wer'], m['levels']['basic']['cer'], m['markers']['τζαι'], m['markers']['εν']
    rows = [('zero-shot Rumble', 'baseline_rumble-2-flash_matched_greedy_dev', 'zeroshot'),
            ('plain LoRA (section 11)', 'rumble-2-flash_lora_greedy_dev', 'lora')]
    for v in VARIANTS:
        if v == 'mix':
            rows.append(('mix, no tag', 'rumble-2-flash_mix_greedy_dev', 'mix_notag'))
        else:
            rows += [(f'{v}: CY tagged / FLEURS untagged', f'rumble-2-flash_{v}_greedy_dev', f'{v}_notag'),
                     (f'{v}: CY untagged / FLEURS tagged', f'rumble-2-flash_{v}_notag_greedy_dev', f'{v}_tag')]
    print(f"\n{'system':34} {'CY WER':>7} {'nodiac':>7} {'CER':>6} | {'FLEURS WER':>10} {'CER':>6} {'τζαι':>5} {'εν':>4}")
    for label, c, f in rows:
        a = cyd(c) if c else None; b = fl(f) if f else None
        sa = f'{a[0]:7.1%} {a[1]:7.1%} {a[2]:6.1%}' if a else f"{'':7} {'':7} {'':6}"
        sb = f'{b[0]:10.1%} {b[1]:6.1%} {b[2]:5} {b[3]:4}' if b else ''
        print(f'{label:34} {sa} | {sb}')
    print('(zero-shot / plain LoRA Cypriot rows: earlier L4 bf16 runs, same settings; Drive runs/)')
finally:
    if AUTO_DISCONNECT:
        from google.colab import runtime
        runtime.unassign()